# Installation

## SWI-Prolog

SWI-Prolog is required to run LangPro. During the installation you will have to press ENTER to continue installing swi-prolog.

In [ ]:
#!sudo apt-get install software-properties-common
!sudo apt-add-repository -y ppa:swi-prolog/stable
!sudo apt-get update
!sudo apt-get install swi-prolog

In [11]:
# test whether swi-prolog is installed and check its version
! swipl --version

SWI-Prolog version 9.0.4 for x86_64-linux


## LangPro
Natural Tableau-based theorem prover that can operate on parsed sentences and detect semantic relations between a set of premises and a hypothesis. While creating this notebook, the `nl` branch of the LangPro repo is most up to date and stable.

In [ ]:
#! git clone https://github.com/kovvalsky/LangPro.git
#! git clone --single-branch --branch nl https://github.com/kovvalsky/LangPro.git
# just to make sure it points the specific commit on which the notebook was tested
#! cd LangPro; git reset --hard dfc0a00f46240e80675139089afdb82cab112332
#! cd LangPro; git pull
! git clone --single-branch --branch nl https://github.com/kovvalsky/LangPro.git

# EasyCCG

## C&C tools
C&C tools ([Clark&Curran, 2007](https://www.aclweb.org/anthology/J07-4004.pdf)) contain POS tagger, named-entity recognizer (NER), CCG parser, and Boxer. We need only the POS tagger, NER and parser. If the data doesn't contain named entities, the NER is irrelevant. It comes with only with the 🇬🇧 English models. As an input, LangPro requires the prolog format of the CCG derivation trees (hence, `--candc-printer boxer`).

In [ ]:
! git clone https://github.com/chrzyki/candc.git

In [ ]:
! candc/candc/bin/candc --version

In [ ]:
! tar -xzf candc/models/models-1.02.tgz -C candc/models

In [ ]:
# test that C&C tools (namely, POS tagger, NER and parser) are working
! echo "Parse this sentence for me" | ../candc/candc/bin/candc --models ../candc/models/models --candc-printer boxer

## Preparing Data for LangPro

LangPro requires two prolog files per dataset: a `*_sen.pl` file recording NLI problems with labels, and a `*_ccg.pl` file containing parse trees for all sentences. For each dataset in `datasets/`, we create those files under `langpro-datasets/` so parsing and theorem proving can be reused without repeated conversions.

The JSON datasets here contain premise sentences like `P1`, `P2`, etc., and a conclusion `C`. We write one `sen_id` entry per premise and one for the conclusion, then parse every sentence from the corresponding `.spl` file.

For each dataset file, create a tokenized sentence-per-line file and a LangPro prolog file. The parser expects tokenized input, so we use NLTK's TreebankWordTokenizer here, which avoids requiring downloaded punkt data.

In [ ]:
from nltk.tokenize import TreebankWordTokenizer
_tokenizer = TreebankWordTokenizer()


✘ No compatible package found for 'en_ner_craft_md' (spaCy v3.8.14)



SystemExit: 1

In [24]:
import re
ENTITY_PATTERNS = [r'member of (.+? pathway)', r'(Gene .+?)\b']
def group_entities_cheat(stmnt):
    result = stmnt
    matches = []
    for pattern in ENTITY_PATTERNS:
        for m in re.finditer(pattern, stmnt):
            matches.append((m.start(1), m.end(1)))
    for start, end in sorted(matches, reverse=True):
        underscored = stmnt[start:end].strip().replace(" ", "_")
        result = result[:start] + underscored + result[end:]

    return result

In [25]:
print(group_entities_cheat("Every member of ABC transporter disorders pathway is a member of Disorders of transmembrane transporters pathway."))
print(group_entities_cheat("Gene ABCD is a member of ABC transporter disorders pathway."))
print(group_entities_cheat("It is true that Gene ABCD is a member of Disorders of transmembrane transporters pathway."))


Every member of ABC_transporter_disorders_pathway is a member of Disorders_of_transmembrane_transporters_pathway.
Gene_ABCD is a member of ABC_transporter_disorders_pathway.
It is true that Gene_ABCD is a member of Disorders_of_transmembrane_transporters_pathway.


In [ ]:
import glob
import json
import os
from nltk.tokenize import TreebankWordTokenizer

def escape_prolog(text):
    return text.replace("'", r"\\'")

os.makedirs("langpro-re-datasets", exist_ok=True)
json_paths = sorted(glob.glob("../datasets/*.json"))
print(f"Found {len(json_paths)} dataset files in ../datasets/")

_tokenizer = TreebankWordTokenizer()

for json_path in json_paths:
    basename = os.path.splitext(os.path.basename(json_path))[0]
    sen_path = os.path.join("langpro-re-datasets", f"{basename}_sen.pl")
    spl_path = os.path.join("langpro-re-datasets", f"{basename}.spl")

    with open(json_path, encoding="utf-8") as f:
        problems = json.load(f)

    with open(sen_path, "w", encoding="utf-8") as sen_f, open(spl_path, "w", encoding="utf-8") as spl_f:
        for idx, problem in enumerate(problems, start=1):
            pid = f"{basename}_{idx}"

            premise_keys = sorted([k for k in problem.keys() if k.startswith('P')])
            premises = [problem[k] for k in premise_keys]
            premises = [group_entities_cheat(premise) for premise in premises]
            
            hypothesis = problem["C"]
            hypothesis = group_entities_cheat(hypothesis)


            # datasets are always generated as entailment in SylloBio-NLI framework, non-entailment is by mixing premises and conclusion at  eval time
            label = "yes"

            sen_f.write(f"% problem id = {pid}\n")

            num_premises = len(premises)
            for i, premise in enumerate(premises):
                sen_id = (num_premises + 1) * (idx - 1) + (i + 1)
                sen_f.write(f"sen_id({sen_id}, '{pid}', 'p', '{label}', '{escape_prolog(premise)}').\n")
                spl_f.write(" ".join(_tokenizer.tokenize(premise)) + "\n")

            hyp_id = (num_premises + 1) * idx
            sen_f.write(f"sen_id({hyp_id}, '{pid}', 'h', '{label}', '{escape_prolog(hypothesis)}').\n")
            spl_f.write(" ".join(_tokenizer.tokenize(hypothesis)) + "\n")

    print(f"Wrote {basename}_sen.pl and .spl")

Found 36 dataset files in ../datasets/
Wrote disjunctive_syllogism-2-0-base-real_dataset_sen.pl and .spl
Wrote disjunctive_syllogism-2-0-complex_predicates-real_dataset_sen.pl and .spl
Wrote disjunctive_syllogism-2-0-de_morgan-real_dataset_sen.pl and .spl
Wrote disjunctive_syllogism-2-0-negation-real_dataset_sen.pl and .spl
Wrote gen_contraposition-2-0-base-real_dataset_sen.pl and .spl
Wrote gen_contraposition-2-0-complex_predicates-real_dataset_sen.pl and .spl
Wrote gen_contraposition-2-0-de_morgan-real_dataset_sen.pl and .spl
Wrote gen_contraposition-2-0-negation-real_dataset_sen.pl and .spl
Wrote gen_dilemma-2-0-base-real_dataset_sen.pl and .spl
Wrote gen_dilemma-2-0-complex_predicates-real_dataset_sen.pl and .spl
Wrote gen_dilemma-2-0-de_morgan-real_dataset_sen.pl and .spl
Wrote gen_dilemma-2-0-negation-real_dataset_sen.pl and .spl
Wrote gen_modus_ponens-2-0-base-dummy_dataset_sen.pl and .spl
Wrote gen_modus_ponens-2-0-base-real_dataset_sen.pl and .spl
Wrote gen_modus_ponens-2-0-co

Parsing the sentences with C&C tools. Stats and progress are written to `langpro-datasets/parsing.log` to avoid buffering large volumes of C&C output in the notebook. Already-parsed files are skipped so the cell is safe to re-run after a crash.

In [1]:
import os
import glob
import subprocess
import time

spl_files = sorted(glob.glob("langpro-re-datasets/*.spl"))
print(f"Found {len(spl_files)} .spl files to parse.")

total_start = time.time()
parsed_count = 0

with open("langpro-re-datasets/parsing.log", "w") as log:
    for spl_path in spl_files:
        basename = os.path.basename(spl_path)
        output_path = os.path.join("langpro-re-datasets", basename.replace(".spl", "_easy_ccg.pl"))

        if os.path.exists(output_path):
            print(f"SKIP  {basename} (already parsed)")
            continue
        
        cc_command = [
            "candc/candc/bin/candc",
            "--models", "candc/models/models",
            "--output", output_path,
            "--candc-printer", "boxer",
            "--candc-parser-noisy_rules=false"
        ]

        easyccg_command = [
            "java", "-jar", "easyccg/easyccg.jar"
            "--model", "model"
        ]
    
        t0 = time.time()
        with open(spl_path, "r") as f_in:
            result = subprocess.run(easyccg_command, stdin=f_in, stderr=log, stdout=subprocess.DEVNULL)
        elapsed = time.time() - t0
        parsed_count += 1

        status = "OK" if result.returncode == 0 else f"FAILED (exit {result.returncode})"
        print(f"[{parsed_count:>3}/{len(spl_files)}] {basename}: {status} ({elapsed:.1f}s)")

total_elapsed = time.time() - total_start
mins, secs = divmod(int(total_elapsed), 60)
print(f"\nParsed {parsed_count} files in {mins}m {secs}s (avg {total_elapsed/parsed_count:.1f}s/file)" if parsed_count else "\nNothing to parse.")

Found 36 .spl files to parse.
[  1/36] disjunctive_syllogism-2-0-base-real_dataset.spl: FAILED (exit 1) (0.0s)
[  2/36] disjunctive_syllogism-2-0-complex_predicates-real_dataset.spl: FAILED (exit 1) (0.0s)
[  3/36] disjunctive_syllogism-2-0-de_morgan-real_dataset.spl: FAILED (exit 1) (0.0s)
[  4/36] disjunctive_syllogism-2-0-negation-real_dataset.spl: FAILED (exit 1) (0.0s)
[  5/36] gen_contraposition-2-0-base-real_dataset.spl: FAILED (exit 1) (0.0s)
[  6/36] gen_contraposition-2-0-complex_predicates-real_dataset.spl: FAILED (exit 1) (0.0s)
[  7/36] gen_contraposition-2-0-de_morgan-real_dataset.spl: FAILED (exit 1) (0.0s)
[  8/36] gen_contraposition-2-0-negation-real_dataset.spl: FAILED (exit 1) (0.0s)
[  9/36] gen_dilemma-2-0-base-real_dataset.spl: FAILED (exit 1) (0.0s)
[ 10/36] gen_dilemma-2-0-complex_predicates-real_dataset.spl: FAILED (exit 1) (0.0s)
[ 11/36] gen_dilemma-2-0-de_morgan-real_dataset.spl: FAILED (exit 1) (0.0s)
[ 12/36] gen_dilemma-2-0-negation-real_dataset.spl: FAIL

Now we already have all necessary prolog files in `langpro-datasets/` for reasoning with LangPro.  
Note that if some sentence is not parsed, its corresponding `ccg($id,...` term won't be in `*_ccg.pl` file.

In [34]:
! grep -cP "ccg\(\d+" langpro-datasets/*_cc_ccg.pl | wc -l

36


# NLI Proving

The elements of `parList` are described [here](https://github.com/kovvalsky/LangPro/wiki/Using-the-prover). The ones you might want to change are:
* `ral(50)` - rule application limit, which means that less you set there less time will be spend to find a proof and the results might be poor);
* `waif(filename)` - write answers in file. In case you want to have LangPro predictions in a file written.;
* `prprb` - by default LangPro prints problems that were not predicted correctly. This flag forces LangPro to print all the problems.

## Custom dataset proving

When running LangPro, we need to feed it with wordnet files to give it access to some lexical knowledge. Other files that needs to be given are prolog files with NLI problem descriptions and parses.

In [29]:
# Directory where NLI proving judgements will be written
! mkdir -p re-results

In [33]:
# Proving all NLI problems from the generated datasets.
# Adjust the glob pattern or loop over specific files as needed.
import glob
import os
import subprocess
import time

sen_files = sorted(glob.glob("langpro-re-datasets/*_sen.pl"))
print(f"Found {len(sen_files)} datasets to prove.")

gstart = time.time()
with open("proving.log", "w") as log:
    for sen_path in sen_files:
        start = time.time()
        basename = os.path.basename(sen_path).replace("_sen.pl", "")
        ccg_path = sen_path.replace("_sen.pl", "_cc_ccg.pl")
        result_path = os.path.join("re-results", f"{basename}_pred.txt")

        if not os.path.exists(ccg_path):
            print(f"SKIP {basename}: no CCG file found")
            continue

        cmd = [
            "swipl",
            "-g",
            f"parList([prprb, ral(50), aall, wn_ant, wn_sim, wn_der, constchk, waif('{result_path}')]), entail_all, halt",
            "-f",
            "LangPro/prolog/main.pl",
            "LangPro/WNProlog/wn.pl",
            sen_path,
            ccg_path,

        ]

        print(f"Proving {basename}...")
        subprocess.run(cmd, stderr=log, stdout=subprocess.DEVNULL)
        print(f"  -> written to {result_path}, took {time.time() - start:.1f}s")
total_elapsed = time.time() - gstart
print(f"\nProved {len(sen_files)} datasets in {total_elapsed:.1f}s (avg {total_elapsed/len(sen_files):.1f}s/dataset)")

Found 36 datasets to prove.
Proving disjunctive_syllogism-2-0-base-real_dataset...
  -> written to re-results/disjunctive_syllogism-2-0-base-real_dataset_pred.txt, took 106.0s
Proving disjunctive_syllogism-2-0-complex_predicates-real_dataset...
  -> written to re-results/disjunctive_syllogism-2-0-complex_predicates-real_dataset_pred.txt, took 85.3s
Proving disjunctive_syllogism-2-0-de_morgan-real_dataset...
  -> written to re-results/disjunctive_syllogism-2-0-de_morgan-real_dataset_pred.txt, took 67.9s
Proving disjunctive_syllogism-2-0-negation-real_dataset...
  -> written to re-results/disjunctive_syllogism-2-0-negation-real_dataset_pred.txt, took 83.4s
Proving gen_contraposition-2-0-base-real_dataset...
  -> written to re-results/gen_contraposition-2-0-base-real_dataset_pred.txt, took 39.2s
Proving gen_contraposition-2-0-complex_predicates-real_dataset...
  -> written to re-results/gen_contraposition-2-0-complex_predicates-real_dataset_pred.txt, took 83.1s
Proving gen_contraposition-

# Preprocessing with scispacy NER

# Evaluating

In [46]:
results_files = sorted(glob.glob("re-results/*_pred.txt"))
print(f"Accuracies for {len(results_files)} result files.")

for result_file in results_files:
    correct = 0
    total = 0
    with open(result_file, "r") as f:
        lines = f.readlines()
        lines = lines[3:]
        for line in lines:
            total += 1
            if "ENTAILMENT" in line:
                correct += 1
    accuracy = correct / total if total > 0 else 0
    print(f"{result_file.replace('re-results/', '').replace('_pred.txt', '').replace('-2-0', ''):<60}: {accuracy:>8.2f}")

Accuracies for 36 result files.
disjunctive_syllogism-base-real_dataset                     :     0.00
disjunctive_syllogism-complex_predicates-real_dataset       :     0.00
disjunctive_syllogism-de_morgan-real_dataset                :     0.00
disjunctive_syllogism-negation-real_dataset                 :     0.00
gen_contraposition-base-real_dataset                        :     0.00
gen_contraposition-complex_predicates-real_dataset          :     0.00
gen_contraposition-de_morgan-real_dataset                   :     0.00
gen_contraposition-negation-real_dataset                    :     0.00
gen_dilemma-base-real_dataset                               :     0.00
gen_dilemma-complex_predicates-real_dataset                 :     0.00
gen_dilemma-de_morgan-real_dataset                          :     0.00
gen_dilemma-negation-real_dataset                           :     0.00
gen_modus_ponens-base-dummy_dataset                         :     0.83
gen_modus_ponens-base-real_dataset           